# Polyglot-ko 3.8B LoRA Fine-tuning Notebook

이 노트북은 다음 단계를 포함합니다:
1. 패키지 설치 및 임포트  
2. 모델 & 토크나이저 로딩  
3. LoRA 구성 및 모델 래핑  
4. 데이터 로드·전처리 및 디버깅 출력  
5. DataLoader 배치 확인  
6. 디버깅 콜백 정의  
7. TrainingArguments 설정  
8. Trainer 초기화  
9. 학습 실행  
10. 최종 모델 저장  

In [1]:
#from peft.utils.other import prepare_model_for_int8_training

0) 필수 패키지 임포트

In [2]:
import os
import json
import torch
import gc
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    TrainerCallback
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_int8_training
)
from datasets import Dataset
from datasets import logging as datasets_logging


1) 모델 및 토크나이저 로딩

In [3]:
model_name = "/home/remote/Ai_Capstone_Project/SourceCode/polyglot-ko-3.8B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_8bit=True,
    device_map="auto",
    torch_dtype=torch.float16
)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id



The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

2) LoRA 적용

In [4]:
model = prepare_model_for_int8_training(model)
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["query_key_value"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

/home/remote/anaconda3/envs/bnb_env/lib/python3.10/site-packages/peft/utils/other.py:122: FutureWarning: prepare_model_for_int8_training is deprecated and will be removed in a future version. Use prepare_model_for_kbit_training instead.
  warnings.warn(


3) JSONL 로드 & 초기 토크나이징 (Dataset.from_list)

In [5]:
import os
from datasets import load_dataset

# 1) JSONL 로드
ds = load_dataset("json", data_files="data/lora_data.jsonl", split="train")

# 2) 현실적인 최대 길이 지정
MAX_LEN = 1024   # 원하는 값으로 조정하세요 (512, 2048 등)

def preprocess(batch):
    batch_size = len(next(iter(batch.values())))
    prompts, answers = [], []

    for i in range(batch_size):
        if "instruction" in batch and "output" in batch:
            inst, outp = batch["instruction"][i], batch["output"][i]
            p = f"### 질문: {inst}\n\n### 답변: "
            a = outp + (tokenizer.eos_token or "")
        elif "text" in batch:
            txt = batch["text"][i]
            if tokenizer.eos_token and not txt.endswith(tokenizer.eos_token):
                txt += tokenizer.eos_token
            p, a = "", txt
        else:
            vals = [batch[col][i] for col in batch.keys()]
            txt = " ".join(map(str, vals))
            if tokenizer.eos_token and not txt.endswith(tokenizer.eos_token):
                txt += tokenizer.eos_token
            p, a = "", txt

        prompts.append(p)
        answers.append(a)

    # truncation=True + max_length=MAX_LEN
    tok_p = tokenizer(
        prompts,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LEN
    )
    tok_a = tokenizer(
        answers,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LEN
    )

    input_ids, labels = [], []
    for p_ids, a_ids in zip(tok_p["input_ids"], tok_a["input_ids"]):
        input_ids.append(p_ids + a_ids)
        labels.append([-100] * len(p_ids) + a_ids)

    return {"input_ids": input_ids, "labels": labels}

# 3) map 호출
tokenized_ds = ds.map(
    preprocess,
    batched=True,
    batch_size=1000,
    num_proc=os.cpu_count(),
    remove_columns=ds.column_names,
    load_from_cache_file=False,
    desc="Tokenizing with multi-process"
)


Tokenizing with multi-process (num_proc=32):   0%|          | 0/1705420 [00:00<?, ? examples/s]

In [6]:
import json
from transformers import AutoTokenizer
from torch.utils.data import IterableDataset, DataLoader, get_worker_info

# 1) 토크나이저 준비
MODEL_NAME = model_name
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

# 2) IterableDataset 정의
class LoRADataset(IterableDataset):
    def __init__(self, file_path):
        self.file_path = file_path

    def _process_line(self, line):
        sample = json.loads(line)
        # 기존 process_line 로직 그대로 써주세요
        if 'instruction' in sample and 'output' in sample:
            prompt = f"### 질문: {sample['instruction']}\n\n### 답변: "
            answer = sample['output'] + (tokenizer.eos_token or "")
            p_ids = tokenizer(prompt, add_special_tokens=False).input_ids
            a_ids = tokenizer(answer, add_special_tokens=False).input_ids
            return {'input_ids': p_ids + a_ids,
                    'labels':    [-100] * len(p_ids) + a_ids}
        elif 'text' in sample:
            text = sample['text']
            if tokenizer.eos_token and not text.endswith(tokenizer.eos_token):
                text += tokenizer.eos_token
            enc = tokenizer(text, truncation=True)
            return {'input_ids': enc.input_ids, 'labels': enc.input_ids.copy()}
        else:
            merged = " ".join(str(v) for v in sample.values())
            if tokenizer.eos_token and not merged.endswith(tokenizer.eos_token):
                merged += tokenizer.eos_token
            enc = tokenizer(merged, truncation=True)
            return {'input_ids': enc.input_ids, 'labels': enc.input_ids.copy()}

    def __iter__(self):
        worker_info = get_worker_info()
        # worker id 와 전체 worker 수 가져오기 (메인 프로세스면 None)
        if worker_info is None:
            start, step = 0, 1
        else:
            start = worker_info.id
            step = worker_info.num_workers

        with open(self.file_path, 'r', encoding='utf-8') as f:
            for idx, line in enumerate(f):
                # 각 워커가 서로 다른 줄을 처리하도록 분할
                if idx % step == start:
                    yield self._process_line(line)

# 3) DataLoader + 병렬 워커
dataset = LoRADataset("data/lora_data.jsonl")
dataloader = DataLoader(
    dataset,
    batch_size=None,     # 이미 토큰화된 dict가 나옴
    num_workers=os.cpu_count(),       # 사용할 프로세스 개수
    prefetch_factor=2    # 워커당 2배치 미리 읽기
)

# 4) 필요하다면 한 번에 모아서 HF Dataset.from_list 가능
#    (이때는 작은 부분집합만 모아서 사용하세요)
batch = list(dataloader)[:1000]  # 예: 첫 1000개만
train_dataset = Dataset.from_list(batch)


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Asking to tru

In [7]:
for idx in range(3):
    sample = train_dataset[idx]
    input_ids = sample["input_ids"]
    labels    = sample["labels"]

    input_text = tokenizer.decode(input_ids, skip_special_tokens=True)
    label_texts = [
        tokenizer.decode([lid], skip_special_tokens=True) if lid != -100 else "<IGNORE>"
        for lid in labels
    ]

    print(f"=== 샘플 {idx} ===")
    print("▶ 입력 텍스트:")
    print(input_text)
    print("▶ 레이블 토큰 (-100은 무시):")
    print(label_texts[:len(label_texts)])
    print()

=== 샘플 0 ===
▶ 입력 텍스트:
캐릭터 설명: 이 캐릭터는 고대 시대의 중년 귀족 출신 학자입니다. 침착함 성격을 지녔습니다.
상황: 플레이어에게 새로운 임무나 퀘스트를 제안하는 상황입니다. {'text': '반드시 해낼 수 있을 거야!', 'tags': ['희망', '확언'], 'intensity': 5}
▶ 레이블 토큰 (-100은 무시):
['캐릭터', ' 설명', ':', ' 이', ' 캐릭터', '는', ' 고대', ' 시대', '의', ' 중년', ' 귀족', ' 출신', ' 학자', '입니다', '.', ' 침착', '함', ' 성격', '을', ' 지녔', '습니다', '.', '\n', '상황', ':', ' 플레이어', '에게', ' 새로운', ' 임무', '나', ' 퀘', '스트', '를', ' 제안', '하', '는', ' 상황', '입니다', '.', ' ', '{', "'", 't', 'ex', 't', "'", ':', " '", '반드시', ' 해낼', ' 수', ' 있', '을', ' 거', '야', '!', "',", " '", 't', 'ag', 's', "'", ':', ' [', "'", '희망', "',", " '", '확', '언', "'", ']', ',', " '", 'int', 'ens', 'ity', "'", ':', ' 5', '}', '']

=== 샘플 1 ===
▶ 입력 텍스트:
캐릭터 설명: 이 캐릭터는 고대 시대의 중년 귀족 출신 학자입니다. 침착함 성격을 지녔습니다.
상황: 플레이어에게 새로운 임무나 퀘스트를 제안하는 상황입니다. {'text': '언젠가는 빛을 볼 날이 올 거야.', 'tags': ['희망', '예측'], 'intensity': 4}
▶ 레이블 토큰 (-100은 무시):
['캐릭터', ' 설명', ':', ' 이', ' 캐릭터', '는', ' 고대', ' 시대', '의', ' 중년', ' 귀족', ' 출신', ' 학자', '입니다', '.', ' 침착', '함', ' 성격

7) 메모리 클린업 Callback 정의

In [8]:
class MemoryCleanupCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        print("[메모리 정리] GC 및 GPU 캐시 비우기…")
        gc.collect()
        torch.cuda.empty_cache()


8) Trainer 및 학습 인자 설정

In [ ]:
import os
import json
import torch
import multiprocessing as mp
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    default_data_collator,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# --- 환경 설정 ---
mp.set_start_method("fork", force=True)
torch.set_num_threads(os.cpu_count())

# --- 설정값 ---
MODEL_NAME = model_name  # 미리 정의된 모델명 사용
DATA_FILE = "data/lora_data.jsonl"
OUTPUT_DIR = "/home/remote/Ai_Capstone_Project/AIbigdata_link/Fine-tuning-LoRA"
os.makedirs(OUTPUT_DIR, exist_ok=True)
MAX_LEN = 512

# --- LoRA 타겟 모듈 설정 (GPT-NeoX의 attention.dense) ---
target_modules = ["attention.dense"]  # 필요시 다른 모듈도 추가 가능

# --- 8-bit 양자화 모델 로드 (GPU 전용) ---
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True
)
model.gradient_checkpointing_enable()
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

# --- LoRA 어댑터 설정 및 적용 ---
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=target_modules,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)
model = get_peft_model(model, lora_config)

# --- 패딩 토큰 설정 ---
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

# --- 데이터 전처리 함수 정의 ---
def process_sample(sample):
    if 'instruction' in sample and 'output' in sample:
        prompt = f"### 질문: {sample['instruction']}\n\n### 답변: "
        answer = sample['output'] + tokenizer.eos_token
        enc = tokenizer(prompt + answer, truncation=True, max_length=MAX_LEN, padding='max_length')
        prompt_ids = tokenizer(prompt, add_special_tokens=False).input_ids
        answer_ids = tokenizer(answer, add_special_tokens=False).input_ids
        labels = [-100] * len(prompt_ids) + answer_ids
        labels = labels[:MAX_LEN] + [-100] * (MAX_LEN - len(labels))
        enc['labels'] = labels
    elif 'text' in sample:
        enc = tokenizer(sample['text'], truncation=True, max_length=MAX_LEN, padding='max_length')
        enc['labels'] = enc.input_ids.copy()
    else:
        merged = ' '.join(str(v) for v in sample.values())
        enc = tokenizer(merged, truncation=True, max_length=MAX_LEN, padding='max_length')
        enc['labels'] = enc.input_ids.copy()
    return enc

# --- 데이터 길이 계산 및 스트리밍 로드 ---
with open(DATA_FILE, 'r', encoding='utf-8') as f:
    ds_len = sum(1 for _ in f)
raw_ds = load_dataset('json', data_files=DATA_FILE, streaming=True)['train']
train_dataset = raw_ds.map(
    process_sample,
    batched=False,
    remove_columns=raw_ds.column_names
).with_format('torch')

# --- 트레이닝 설정 ---
batch_size = 4
accum_steps = 4
epochs = 3
max_steps = (ds_len // (batch_size * accum_steps)) * epochs

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=accum_steps,
    learning_rate=2e-4,
    fp16=True,
    optim='adamw_torch',
    dataloader_num_workers=2,
    dataloader_pin_memory=False,
    dataloader_drop_last=True,
    dataloader_persistent_workers=True,
    dataloader_prefetch_factor=4,
    logging_steps=10,
    save_strategy='steps',
    #save_steps=2000,
    save_steps=100, #테스트용
    logging_dir=f"{OUTPUT_DIR}/logs",
    report_to='tensorboard',
    num_train_epochs=epochs,
    max_steps=max_steps
)

# --- Trainer 초기화 및 학습 시작 ---
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=default_data_collator,
    tokenizer=tokenizer
)

if __name__ == '__main__':
    trainer.train()


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

/tmp/ipykernel_30118/1303142268.py:122: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
Too many dataloader workers: 2 (max is dataset.n_shards=1). Stopping 1 dataloader workers.
/home/remote/anaconda3/envs/bnb_env/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:316: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss


In [ ]:
# 9) 학습 완료 후 모델·토크나이저 저장
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("✅ 모델과 토크나이저가 저장되었습니다:", OUTPUT_DIR)
